# 13.5 Local Deployment — Colab lab

Companion to the [course page](https://enduringchung.github.io/ml-interactive-course/part-13-large-language-models/13-5-local-deployment.html). Quantization, memory budgets, and the bandwidth-bound decoding model — measured on real weights.

**Runtime → Change runtime type → T4 GPU** (free tier is enough for 1–3B models).


In [ ]:
import torch
print(torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")

## 1. The memory budget, measured

Load a small model in fp16 and measure the actual VRAM against the formula from the page: `VRAM ≈ bytes_per_weight × N + KV-cache + overhead`.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"   # small enough for the free T4 (16 GiB)
tok = AutoTokenizer.from_pretrained(model_name)

def load_and_measure(dtype, label):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype,
                                                 device_map="cuda")
    n_params = sum(p.numel() for p in model.parameters())
    weights_gib = n_params * (2 if dtype == torch.float16 else 8) / 2**30
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f"{label:>10}: {n_params/1e6:.0f}M params, weights {weights_gib:.2f} GiB, "
          f"peak VRAM {peak:.2f} GiB (overhead {peak - weights_gib:.2f} GiB)")
    return model

m16 = load_and_measure(torch.float16, "fp16")

## 2. Quantization: int8 and int4 with bitsandbytes

Same model, same weights, fewer bits. The page's per-group story in one line: `load_in_4bit` uses block-wise scales.


In [ ]:
from transformers import BitsAndBytesConfig

m8 = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=BitsAndBytesConfig(load_in_8bit=True),
    device_map="cuda")
m4 = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=BitsAndBytesConfig(load_in_4bit=True),
    device_map="cuda")

for m, label in [(m8, "int8"), (m4, "int4")]:
    n = sum(p.numel() for p in m.parameters())
    print(f"{label}: {n/1e6:.0f}M params -> ~{n * (1 if label=='int8' else 0.5) / 2**30:.2f} GiB weights")

## 3. Quality vs size: perplexity on a held-out snippet

The page's claim: int4 costs ~0.1–0.3 perplexity. Measure it — same text, three precisions.


In [ ]:
import math

test_text = ("The transformer architecture replaces recurrence with self-attention, "
             "allowing the model to process all tokens in parallel during training.")

@torch.no_grad()
def perplexity(model, text, max_len=64):
    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_len).to("cuda")
    out = model(**enc, labels=enc["input_ids"])
    return math.exp(out.loss.item())

for m, label in [(m16, "fp16"), (m8, "int8"), (m4, "int4")]:
    print(f"{label}: perplexity {perplexity(m, test_text):.2f}")

## 4. The bandwidth-bound speed model

Time generation for each precision. The page's arithmetic: tokens/s ≈ bandwidth / bytes-per-token. Check how close the free T4 gets.


In [ ]:
import time

@torch.no_grad()
def bench(model, prompt="The best deployment is", steps=50):
    enc = tok(prompt, return_tensors="pt").to("cuda")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    model.generate(**enc, max_new_tokens=steps, do_sample=False)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return steps / dt

for m, label in [(m16, "fp16"), (m8, "int8"), (m4, "int4")]:
    tps = bench(m)
    n = sum(p.numel() for p in m.parameters())
    bytes_per_tok = n * (2 if label == "fp16" else 1 if label == "int8" else 0.5)
    print(f"{label}: {tps:.1f} tokens/s | implied bandwidth "
          f"{tps * bytes_per_tok / 1e9:.0f} GB/s")

## 5. Exercises

1. Load `Qwen/Qwen2.5-1.5B` in int4 and int8. Does the quality gap (Section 3) grow with model size or shrink? Explain why.
2. Rerun Section 4 with `do_sample=True` and a temperature — does the speed change? (Hint: sampling adds a softmax + draw per step, but the weight read dominates.)
3. The KV-cache: time generation with `max_new_tokens=50` vs `max_new_tokens=500` on the int4 model. The *per-token* speed should be nearly constant (bandwidth-bound) — verify, then compute the cache memory at 500 tokens with the page's formula (`2 × L × h_kv × d × T × 2 bytes`).
4. (Local-only) Install Ollama on your laptop, pull `qwen2.5:7b-instruct-q4_K_M`, and compare its tokens/s to this Colab T4 — the page's bandwidth model predicts which wins and why.
